# Economic Activity Areas Near Greenspace

This notebook selects economic activity areas that are within **500 metres of local greenspace** in Montréal. It reads the source GeoJSON files, checks that their coordinate systems use metres, filters the economic areas by distance, and saves the result as a new GeoJSON file.

Run the cells from top to bottom. The final cell writes the filtered areas next to the original economic activity data and checks the saved feature count.

## 1. Load and inspect the economic activity data

The first cell imports the tools, finds the input file from either the notebook folder or its parent, and loads the GeoJSON. It then selects features whose `Affectatio` value is `Activités économiques`.

In [12]:
# Import tools for reading JSON data and working with file paths
import json
from pathlib import Path

# Set the expected location of the data file
input_path = "Data/intens_affect/intens_affect.json"

# Find the folder where the notebook is currently running
base_dir = Path.cwd().resolve()

# Check likely locations for the data file
candidate_paths = [
    base_dir / "Data" / "intens_affect" / "intens_affect.json",
    base_dir.parent / "Data" / "intens_affect" / "intens_affect.json",
]

# Use the first location where the data file exists
input_path = next((p for p in candidate_paths if p.exists()), candidate_paths[0])

# Open the JSON file and load its contents into Python
with open(input_path, encoding="utf-8") as f:
    data = json.load(f)

# Show how many features are in the file and the properties of the first feature
print("Number of features:", len(data["features"]))
print(data["features"][0]["properties"])

# Create an empty list to hold only economic activity areas
economic_activities = []

# Go through each feature and keep those marked as economic activities
for feature in data["features"]:
    if feature["properties"]["Affectatio"] == "Activités économiques":
        economic_activities.append(feature)

# Show how many economic activity areas were found
print("Economic activity areas:", len(economic_activities))

Number of features: 716
{'NiveauINT': 99.0, 'Niv_txt': 'Non applicable', 'Densite': 0.0, 'Affectatio': 'Conservation', 'Shape_Area': 247383.812516}
Economic activity areas: 27


## 2. Keep areas within 500 m of greenspace

Both datasets must use **EPSG:32188**, a projected coordinate system measured in metres, so a 500 m buffer represents the intended distance. The cell combines the greenspace geometries, buffers them by 500 m, and keeps each economic activity area that intersects that buffer.

In [13]:
# Load the geometry helpers used to combine and test the polygons
from shapely import union_all
from shapely.geometry import shape

# Load local greenspace polygons from the matching projected dataset
base_dir = Path.cwd().resolve()
greenspace_candidates = [
    base_dir / "Data" / "affect_espacevertlocal" / "affect_espacevertlocal.json",
    base_dir.parent / "Data" / "affect_espacevertlocal" / "affect_espacevertlocal.json",
]
greenspace_path = next((p for p in greenspace_candidates if p.exists()), greenspace_candidates[0])
with open(greenspace_path, encoding="utf-8") as f:
    greenspace_data = json.load(f)

# Confirm both datasets use the projected CRS whose units are metres
expected_crs = "urn:ogc:def:crs:EPSG::32188"
economic_crs = data["crs"]["properties"]["name"]
greenspace_crs = greenspace_data["crs"]["properties"]["name"]
if economic_crs != expected_crs or greenspace_crs != expected_crs:
    raise ValueError("Economic areas and greenspace must both use EPSG:32188 for a 500 m distance filter.")

# Merge greenspace polygons, then make a 500 m zone around them
greenspace_union = union_all([shape(feature["geometry"]) for feature in greenspace_data["features"]])
greenspace_500m_buffer = greenspace_union.buffer(500)

# Keep an area if it touches the 500 m zone around any greenspace
all_economic_activities = economic_activities.copy()
economic_activities = [
    feature
    for feature in all_economic_activities
    if shape(feature["geometry"]).intersects(greenspace_500m_buffer)
]

print("Economic activity areas within 500 m of greenspace:", len(economic_activities))

Economic activity areas within 500 m of greenspace: 24


## 3. Save and check the filtered areas

The last cell keeps the original GeoJSON metadata, replaces its feature list with the filtered areas, and writes the result beside the source file. It then reopens the saved file and checks that all 24 expected areas were written.

In [14]:
# Save the economic areas within 500 m of greenspace
output_path = input_path.parent / "economic_areas_within_500m.geojson"

# Keep the original GeoJSON structure and replace its features
output_data = data.copy()
output_data["features"] = economic_activities

# Write the results to a GeoJSON file
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output_data, f)

# Confirm the output was saved correctly
print("GeoJSON saved to:", output_path)
print("Economic areas saved:", len(output_data["features"]))

# Check the saved file
with open(output_path, encoding="utf-8") as f:
    saved_data = json.load(f)

assert len(saved_data["features"]) == 24
print("Check passed: 24 economic areas saved.")

GeoJSON saved to: /Users/gianniscaramuzzi/GEOG 464/A2_Scaramuzzi/Data/intens_affect/economic_areas_within_500m.geojson
Economic areas saved: 24
Check passed: 24 economic areas saved.


## 4. Select building roof footprints

This section implements `process_building_footprints.py` directly in the notebook. It downloads Montreal's building-roof shapefile, verifies EPSG:32188 and the Montréal MTM zone 8 projection, keeps roofs that overlap economic zones by at least 50%, calculates each roof's distance to local greenspace, and saves the results to `Data/buildings/building_footprints_in_economic_zones.geojson`.

In [ ]:
# Import tools for downloading and processing the building footprint shapefile
import json
import re
import shutil
import subprocess
import tempfile
import zipfile
from pathlib import Path

import shapefile
from shapely.geometry import mapping, shape
from shapely.ops import unary_union

# Find the A2_Scaramuzzi project folder from a notebook or repository working directory
cwd = Path.cwd().resolve()
project_candidates = [
    cwd,
    cwd / "A2_Scaramuzzi",
    cwd.parent,
    cwd.parent / "A2_Scaramuzzi",
]
project_dir = next(
    (p for p in project_candidates if (p / "Data" / "intens_affect").exists()),
    cwd,
)

building_data_url = (
    "https://donnees.montreal.ca/dataset/"
    "fab160ae-c81d-46f8-8f92-4a01c10d4390/resource/"
    "183866d5-9027-456a-b0af-36b2e45c8aac/download/"
    "batiments_2d_2016_arrondissements.zip"
)
building_layer = "CARTO-BAT-TOIT"
expected_crs = "urn:ogc:def:crs:EPSG::32188"
minimum_zone_overlap = 0.5
greenspace_distance_metres = 500
economic_path = project_dir / "Data" / "intens_affect" / "economic_activities.geojson"
greenspace_path = project_dir / "Data" / "affect_espacevertlocal" / "affect_espacevertlocal.json"
output_path = project_dir / "Data" / "buildings" / "building_footprints_in_economic_zones.geojson"


def download_source(destination):
    """Download the Montreal building archive to a temporary file."""
    subprocess.run(
        [
            "curl", "--fail", "--location", "--retry", "2",
            "--user-agent", "Mozilla/5.0 (compatible; MontrealBuildingFootprintStudy/1.0)",
            "--referer", "https://donnees.montreal.ca/dataset/batiment-2d",
            "--output", str(destination), building_data_url,
        ],
        check=True,
    )


def extract_roof_layer(archive_path, destination):
    """Extract only the shapefile pieces needed for the roof layer."""
    with zipfile.ZipFile(archive_path) as archive:
        for extension in (".shp", ".shx", ".dbf", ".prj"):
            member_name = building_layer + extension
            try:
                member = archive.getinfo(member_name)
            except KeyError as error:
                raise ValueError(f"Building dataset is missing {member_name}.") from error
            with archive.open(member) as source, (destination / member_name).open("wb") as target:
                shutil.copyfileobj(source, target)


# Load and validate both study layers
with economic_path.open(encoding="utf-8") as source:
    economic_data = json.load(source)
with greenspace_path.open(encoding="utf-8") as source:
    greenspace_data = json.load(source)

for label, dataset in (("Economic zones", economic_data), ("Greenspaces", greenspace_data)):
    crs_name = dataset.get("crs", {}).get("properties", {}).get("name")
    if crs_name != expected_crs:
        raise ValueError(f"{label} must use EPSG:32188; found {crs_name!r}.")

# Combine polygons so every roof is checked against one economic-zone shape
# and one greenspace shape.
economic_zones = unary_union([shape(feature["geometry"]) for feature in economic_data["features"]])
greenspaces = unary_union([shape(feature["geometry"]) for feature in greenspace_data["features"]])

with tempfile.TemporaryDirectory(prefix="montreal-buildings-") as temporary_directory:
    temporary_path = Path(temporary_directory)
    archive_path = temporary_path / "building_footprints.zip"
    download_source(archive_path)
    extract_roof_layer(archive_path, temporary_path)

    projection = (temporary_path / f"{building_layer}.prj").read_text(encoding="utf-8")
    if not re.search(r'Central_Meridian",\s*-73\.5', projection, re.IGNORECASE):
        raise ValueError("Building dataset projection is not Montréal MTM zone 8.")
    if not re.search(r'False_Easting",\s*304800', projection, re.IGNORECASE):
        raise ValueError("Building dataset projection does not match EPSG:32188.")

    reader = shapefile.Reader(str(temporary_path / building_layer), encoding="latin1")
    features = []
    within_distance_count = 0

    for source_id, shape_record in enumerate(reader.iterShapeRecords():
        roof = shape(shape_record.shape.__geo_interface__)
        if roof.is_empty or roof.area <= 0 or not roof.intersects(economic_zones):
            continue

        overlap_ratio = roof.intersection(economic_zones).area / roof.area
        if overlap_ratio < minimum_zone_overlap:
            continue

        distance = roof.distance(greenspaces)
        within_500m = distance <= greenspace_distance_metres
        features.append({
            "type": "Feature",
            "id": source_id,
            "properties": {
                "source_id": source_id,
                "roof_area_m2": round(roof.area, 1),
                "economic_zone_overlap_percent": round(overlap_ratio * 100, 1),
                "greenspace_distance_m": round(distance, 1),
                "within_500m": within_500m,
            },
            "geometry": mapping(roof),
        })
        within_distance_count += within_500m

output_data = {
    "type": "FeatureCollection",
    "name": "building_roof_footprints_in_economic_zones",
    "crs": {"type": "name", "properties": {"name": expected_crs}},
    "features": features,
}
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as output:
    json.dump(output_data, output, separators=(",", ":"))

print(f"Building roof outlines in economic zones: {len(features)}")
print(f"Within 500 m of greenspace: {within_distance_count}")
print(f"Over 500 m from greenspace: {len(features) - within_distance_count}")
print(f"Saved: {output_path}")